In [9]:
import pandas as pd

# Load raw data
customers = pd.read_csv("../data/customers.csv")
products =  pd.read_csv("../data/products.csv")
orders =  pd.read_csv("../data/orders.csv")
stores =  pd.read_csv("../data/stores.csv")

#cleaning column names
def clean_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )
    return df

customers = clean_columns(customers)
products = clean_columns(products)
orders = clean_columns(orders)
stores = clean_columns(stores)

#removing duplicate rows
customers = customers.drop_duplicates()
products = products.drop_duplicates()
orders = orders.drop_duplicates()
stores = stores.drop_duplicates()

#cleaning customers table
customers['signup_date'] = pd.to_datetime(customers['signup_date'], format="%d/%m/%Y", errors='coerce') #converst signup_date from string to datetime using format and turns invalid dates into Nat
customers['signup_date'] = customers['signup_date'].ffill() #forward fill replaces invalid date with previous valid date

customers['gender'] = customers['gender'].fillna("Unknown")
customers['age_group'] = customers['age_group'].fillna("Unknown")
customers['city'] = customers['city'].fillna("Unknown")
customers['loyalty_status'] = customers['loyalty_status'].fillna("Bronze")

# cleaning Products table
products['category'] = products['category'].fillna("Uncategorized")
products['subcategory'] = products['subcategory'].fillna("Misc")
products['price'] = products['price'].fillna(products['price'].median())
products['cost'] = products['cost'].fillna(products['cost'].median())
products['stock_level'] = products['stock_level'].fillna(0)

#cleaning orders table
orders['quantity'] = orders['quantity'].fillna(1)
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
orders['order_date'] = orders['order_date'].ffill()
orders['payment_method'] = orders['payment_method'].fillna("Unknown")
orders['order_status'] = orders['order_status'].fillna("Unknown")

# cleaning Stores table
stores['city'] = stores['city'].fillna("Unknown")
stores['manager'] = stores['manager'].fillna("Unknown")
stores['footfall'] = stores['footfall'].fillna(stores['footfall'].median())
stores['store_type'] = stores['store_type'].fillna("High Street")

#validating 
valid_customers= orders['customer_id'].isin(customers['customer_id'])
valid_products= orders['product_id'].isin(products['product_id'])
valid_stores= orders['store_id'].isin(stores['store_id'])

orders = orders[valid_customers & valid_products & valid_stores]

#Resetting indexes after filtering and dropping duplicates

customers.reset_index(drop=True, inplace=True)
products.reset_index(drop=True, inplace=True)
orders.reset_index(drop=True, inplace=True)
stores.reset_index(drop=True, inplace=True)

#final summary and preview

print("Customers:", customers.shape)
print("Products:", products.shape)
print("Orders:", orders.shape)
print("Stores:", stores.shape)

customers.head(), products.head(), orders.head(), stores.head()



Customers: (3000, 6)
Products: (790, 6)
Orders: (80, 8)
Stores: (8, 5)


(  customer_id signup_date    gender age_group  city loyalty_status
 0       C0001         NaT   Johnson    Female    34         London
 1       C0002         NaT  Williams      Male    29     Manchester
 2       C0003         NaT     Brown    Female    41     Birmingham
 3       C0004         NaT     Jones      Male    22          Leeds
 4       C0005         NaT    Miller    Female    27        Bristol,
   product_id category subcategory  price  cost  stock_level
 0      P0001      Men    T-Shirts     25    12          120
 1      P0002      Men    T-Shirts     28    14          140
 2      P0003      Men    T-Shirts     22    10          160
 3      P0004      Men    T-Shirts     30    15          110
 4      P0005      Men    T-Shirts     27    13          130,
   order_id customer_id product_id  quantity order_date store_id  \
 0    O0001       C0001      P0001         2 2024-03-01     S001   
 1    O0002       C0002      P0004         1 2024-04-01     S001   
 2    O0003       C0